# 3. Digital Twin Fitting & Validation

Create and validate the digital twin simulator.

## 3.0 Install Dependencies

In [1]:
# Install required packages (safe to re-run; --quiet suppresses noise)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'pandas',
    'pyarrow',
    'numpy',
    'scipy',
    'simpy',
    'scikit-learn',
    'joblib',
    'matplotlib',
    'seaborn'
])


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


0

## 3.1 Setup

In [2]:
import sys; from pathlib import Path
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / 'src' / 'data_ingestion.py').exists():
        _repo_root = _p
        _src = str(_p / 'src')
        if _src not in sys.path: sys.path.insert(0, _src)
        break
DATASET = "BPIC2012"
OUTPUT_DIR = _repo_root / 'output' / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3.1 Load Data

In [3]:
import pandas as pd
df = pd.read_parquet(OUTPUT_DIR / f'events_{DATASET}_train.parquet')
print(f"Loaded {len(df)} events")

Loaded 262200 events


## 3.2 Fit Twin

> The fitted twin is used by `ProcessEnv` which now exposes a `MultiDiscrete([max_successors, 15])` action space — routing decisions combined with 15 KPI-based management actions (see `src/kpi_actions.py`).

In [4]:
import joblib
from digital_twin import DigitalTwin
twin = DigitalTwin(max_trace_len=200, seed=42)
twin.fit(df)
joblib.dump(twin, OUTPUT_DIR / f'digital_twin_{DATASET}_train.pkl')
print(f"Fitted twin with {len(twin.activities)} activities")

Fitted twin with 24 activities


## 3.3 Simulate

In [5]:
sim_df = twin.simulate(n_cases=df['case_id'].nunique())
sim_df.to_parquet(OUTPUT_DIR / 'sim_events_modeA.parquet', engine='pyarrow', index=False)
print(f"Generated {len(sim_df)} events")

Generated 252009 events


## 3.4 Validate

In [6]:
import json
from validation import validate, DEFAULT_THRESHOLDS
results = validate(df, sim_df, DEFAULT_THRESHOLDS)
with open(OUTPUT_DIR / 'validation_modeA.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Validation:', results.get('overall_pass', False))

  ✓ PASS  trace_length_wasserstein            = 0.7828  (threshold ≤ 3.0)
  ✓ PASS  case_duration_wasserstein           = 0.0349  (threshold ≤ 0.15)
  ✓ PASS  activity_freq_jsd                   = 0.0014  (threshold ≤ 0.05)
  ✓ PASS  transition_matrix_l1                = 0.0017  (threshold ≤ 0.1)
  ✓ PASS  duration_distribution_ks            = 0.0112  (threshold ≤ 0.2)
  ✓ PASS  variant_coverage                    = 1.0000  (threshold ≥ 0.8)
  ✓ PASS  resource_utilisation_mae            = 0.0002  (threshold ≤ 0.05)

  Overall: ✓ PASS
Validation: True


## 3.5 Validation Metrics Reference

| Metric | What it measures | Scale | Direction | Threshold |
|---|---|---|---|---|
| `trace_length_wasserstein` | Earth Mover's Distance between real and simulated trace length distributions. Penalises simulators that generate traces that are too short or too long on average. | Events (unbounded) | Lower is better | < 3.0 |
| `case_duration_wasserstein` | EMD between normalised [0,1] case duration shapes. Compares the *shape* of duration distributions rather than absolute scale, since the sim captures processing time while real data includes waiting time. | [0, 1] | Lower is better | < 0.15 |
| `activity_freq_jsd` | Jensen-Shannon Divergence on activity frequency distributions. Measures whether the sim generates activities in the same proportions as the real log. JSD is symmetric and bounded. | [0, 1] | Lower is better | < 0.05 |
| `transition_matrix_l1` | Mean L1 distance per cell of the empirical transition matrices. Checks whether the sim routes cases through the same activity sequences as the real process. | [0, 1] | Lower is better | < 0.10 |
| `duration_distribution_ks` | Mean Kolmogorov-Smirnov statistic on per-activity duration CDFs (log-scale). Measures how well the sim reproduces the duration distribution of each individual activity. | [0, 1] | Lower is better | < 0.20 |
| `variant_coverage` | Fraction of activity bigrams from the top-50 most frequent real traces that appear in the simulation. Tests structural coverage of common process paths. | [0, 1] | Higher is better | > 0.80 |
| `resource_utilisation_mae` | Mean absolute error on per-resource utilisation fractions. Checks whether the sim assigns work to resources in the same proportions as the real log. | [0, 1] | Lower is better | < 0.05 |

> Thresholds are informed by empirical ranges from Chapela-Campa et al. (2023) *"Can I Trust My Simulation Model?"* and the Simod/Prosimos benchmarks on BPIC event logs.

## 3.6 Validation Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json

# ── Load results ──────────────────────────────────────────────────────────────
with open(OUTPUT_DIR / 'validation_modeA.json', 'r') as f:
    validation = json.load(f)

report = pd.DataFrame([
    {'metric': k, 'value': v['value'], 'threshold': v['threshold'], 'passed': v['passed']}
    for k, v in validation.items() if k != 'overall_pass'
])
overall = validation.get('overall_pass', False)

# ── Friendly short labels ─────────────────────────────────────────────────────
LABELS = {
    'trace_length_wasserstein':  'Trace Length\nWasserstein',
    'case_duration_wasserstein': 'Case Duration\nWasserstein',
    'activity_freq_jsd':         'Activity Freq\nJSD',
    'transition_matrix_l1':      'Transition\nMatrix L1',
    'duration_distribution_ks':  'Duration\nDist. KS',
    'variant_coverage':          'Variant\nCoverage',
    'resource_utilisation_mae':  'Resource\nUtil. MAE',
}
# Higher-is-better metrics (threshold is a minimum)
HIGHER_BETTER = {'variant_coverage'}

report['label'] = report['metric'].map(LABELS)

# ── Colour palette ────────────────────────────────────────────────────────────
PASS_COLOR  = '#2ecc71'   # green
FAIL_COLOR  = '#e74c3c'   # red
VALUE_ALPHA = 0.85
THRESH_COLOR = '#95a5a6'  # muted grey

# ── Figure layout ─────────────────────────────────────────────────────────────
sns_style = {'axes.facecolor': '#f8f9fa', 'figure.facecolor': 'white',
             'axes.grid': True, 'grid.color': '#dee2e6', 'grid.linewidth': 0.6}
plt.rcParams.update(sns_style)

fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 1, hspace=0.55)

# ── Top panel: value vs threshold (horizontal bars) ───────────────────────────
ax_top = fig.add_subplot(gs[0])
n = len(report)
y = np.arange(n)
bar_h = 0.35

bar_colors = [PASS_COLOR if p else FAIL_COLOR for p in report['passed']]
bars = ax_top.barh(y + bar_h/2, report['value'], bar_h,
                   color=bar_colors, alpha=VALUE_ALPHA, label='Metric value', zorder=3)
ax_top.barh(y - bar_h/2, report['threshold'], bar_h,
            color=THRESH_COLOR, alpha=0.55, label='Threshold', zorder=3)

# Annotate values on bars
for bar, val in zip(bars, report['value']):
    ax_top.text(bar.get_width() + max(report['value']) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', ha='left', fontsize=8, color='#2c3e50')

ax_top.set_yticks(y)
ax_top.set_yticklabels(report['label'], fontsize=9)
ax_top.set_xlabel('Score', fontsize=10)
ax_top.set_title('Metric Value vs Threshold', fontsize=11, fontweight='bold', pad=8)
ax_top.legend(loc='lower right', fontsize=9)
ax_top.invert_yaxis()
ax_top.set_xlim(0, max(report[['value','threshold']].max()) * 1.25)

# ── Bottom panel: pass/fail scorecard ─────────────────────────────────────────
ax_bot = fig.add_subplot(gs[1])
x = np.arange(n)
scorecard_colors = [PASS_COLOR if p else FAIL_COLOR for p in report['passed']]
ax_bot.bar(x, [1] * n, color=scorecard_colors, alpha=0.8, width=0.5, zorder=3)

for xi, (_, row) in zip(x, report.iterrows()):
    symbol = '✓' if row['passed'] else '✗'
    ax_bot.text(xi, 0.5, symbol, ha='center', va='center',
                fontsize=20, fontweight='bold', color='white')

ax_bot.set_xticks(x)
ax_bot.set_xticklabels(report['label'], fontsize=9)
ax_bot.set_yticks([])
ax_bot.set_ylim(0, 1.4)
ax_bot.set_title('Pass / Fail Scorecard', fontsize=11, fontweight='bold', pad=8)

pass_patch = mpatches.Patch(color=PASS_COLOR, alpha=0.8, label='Pass')
fail_patch = mpatches.Patch(color=FAIL_COLOR, alpha=0.8, label='Fail')
ax_bot.legend(handles=[pass_patch, fail_patch], loc='upper right', fontsize=9)

# ── Super-title ───────────────────────────────────────────────────────────────
status_text = '✓  OVERALL PASS' if overall else '✗  OVERALL FAIL'
status_color = PASS_COLOR if overall else FAIL_COLOR
fig.suptitle(f'Digital Twin Validation — {status_text}',
             fontsize=13, fontweight='bold', color=status_color, y=1.01)

plt.savefig(OUTPUT_DIR / 'validation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Overall validation passed: {overall}')
print(f'Saved to {OUTPUT_DIR / "validation_results.png"}')